# 2. Modelado de Machine Learning
## Laboratorio de Ciencia de Datos ADA — EPN
Construcción de un modelo de regresión para predecir el precio de alquiler.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib, json, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

## 1. Carga y Preparación

In [ ]:
df = pd.read_csv('../data/real_state_clean.csv')
print(f"Dataset: {df.shape}")

FEATURES = ['Provincia', 'Lugar', 'Num. dormitorios', 'Num. banos', 'Area', 'Num. garages']
TARGET = 'Precio'

# Agrupar lugares con pocas muestras en 'Otro'
MIN_SAMPLES = 3
conteo = df['Lugar'].value_counts()
lugares_raros = conteo[conteo < MIN_SAMPLES].index
df['Lugar'] = df['Lugar'].apply(lambda x: 'Otro' if x in lugares_raros else x)

X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

## 2. Pipeline de Preprocesamiento

In [ ]:
cat_features = ['Provincia', 'Lugar']
num_features = ['Num. dormitorios', 'Num. banos', 'Area', 'Num. garages']

cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='constant', fill_value='Desconocido')),
    ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
num_pipe = Pipeline([('imp', SimpleImputer(strategy='median'))])

preprocessor = ColumnTransformer([
    ('cat', cat_pipe, cat_features),
    ('num', num_pipe, num_features)
])

## 3. Comparación de Modelos

Se evalúan cuatro algoritmos de regresión:
- **Ridge**: baseline lineal con regularización L2
- **RandomForest**: ensemble de árboles, robusto a outliers
- **GradientBoosting**: boosting secuencial
- **XGBoost**: boosting optimizado, estado del arte en datos tabulares

In [ ]:
modelos = {
    'Ridge':            Ridge(),
    'RandomForest':     RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=200, random_state=42),
    'XGBoost':          XGBRegressor(n_estimators=200, random_state=42, verbosity=0),
}

resultados = {}
print(f"{'Modelo':<22} {'MAE':>8} {'RMSE':>9} {'R²':>8}")
print('-'*50)
for nombre, modelo in modelos.items():
    pipe = Pipeline([('pre', preprocessor), ('model', modelo)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    resultados[nombre] = (mae, rmse, r2)
    print(f"{nombre:<22} {mae:>8.2f} {rmse:>9.2f} {r2:>8.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
nombres = list(resultados.keys())
maes = [v[0] for v in resultados.values()]
cols = ['tomato' if n == 'XGBoost' else 'steelblue' for n in nombres]
ax.bar(nombres, maes, color=cols)
ax.set_title('Comparación de Modelos – MAE (menor es mejor)')
ax.set_ylabel('MAE (USD)')
for i, v in enumerate(maes):
    ax.text(i, v + 3, f'${v:.0f}', ha='center')
plt.tight_layout(); plt.show()

## 4. Modelo Seleccionado: XGBoost

**Justificación:**
- Obtiene el mejor (o comparable) MAE en el conjunto de prueba.
- Maneja nativamente variables categóricas codificadas ordinalmente y valores faltantes.
- No requiere normalización de variables numéricas.
- Altamente configurable (learning rate, profundidad, regularización).
- Ampliamente usado en competencias de ciencia de datos con datos tabulares.

In [ ]:
print("Cross-Validation XGBoost (5-fold, dataset completo):")
best_pipe = Pipeline([
    ('pre', preprocessor),
    ('model', XGBRegressor(n_estimators=300, learning_rate=0.05,
                            max_depth=6, random_state=42, verbosity=0))
])
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_mae = -cross_val_score(best_pipe, X, y, cv=kf, scoring='neg_mean_absolute_error')
cv_r2  =  cross_val_score(best_pipe, X, y, cv=kf, scoring='r2')
print(f"  MAE : {cv_mae.mean():.2f} ± {cv_mae.std():.2f}")
print(f"  R²  : {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")

## 5. Entrenamiento Final y Métricas

In [ ]:
best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("Métricas finales en Test Set:")
print(f"  MAE  = ${mae:.2f}")
print(f"  RMSE = ${rmse:.2f}")
print(f"  R²   = {r2:.4f}")
print(f"  MAPE = {mape:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Evaluación del Modelo XGBoost', fontsize=13, fontweight='bold')

ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.5, color='steelblue', s=20)
lims = [0, max(y_test.max(), y_pred.max()) + 100]
ax.plot(lims, lims, 'r--', lw=1.5)
ax.set_xlabel('Precio Real (USD)'); ax.set_ylabel('Precio Predicho (USD)')
ax.set_title(f'Real vs Predicho  (R²={r2:.3f})')

ax = axes[1]
residuos = y_test - y_pred
ax.hist(residuos, bins=35, color='steelblue', edgecolor='white')
ax.axvline(0, color='red', ls='--')
ax.set_xlabel('Residuo (USD)'); ax.set_title(f'Distribución de Residuos (MAE=${mae:.0f})')
plt.tight_layout(); plt.show()

## 6. Serialización del Modelo

In [ ]:
joblib.dump(best_pipe, '../model/model.pkl')
print("Modelo guardado en ../model/model.pkl")

lugares   = sorted([str(x) for x in df['Lugar'].dropna().unique()])
provincias = sorted([str(x) for x in df['Provincia'].dropna().unique()])
with open('../model/metadata.json', 'w', encoding='utf-8') as f:
    json.dump({'lugares': lugares, 'provincias': provincias}, f, ensure_ascii=False, indent=2)
print("Metadata guardada en ../model/metadata.json")

## 7. Predicciones de Ejemplo

In [ ]:
ejemplos = pd.DataFrame([
    {'Provincia':'Pichincha','Lugar':'Quito',     'Num. dormitorios':3,'Num. banos':2.0,'Area':120.0,'Num. garages':1.0},
    {'Provincia':'Pichincha','Lugar':'Sangolquí', 'Num. dormitorios':2,'Num. banos':1.0,'Area':80.0, 'Num. garages':0.0},
    {'Provincia':'Guayas',   'Lugar':'Guayaquil', 'Num. dormitorios':3,'Num. banos':2.0,'Area':100.0,'Num. garages':1.0},
    {'Provincia':'Pichincha','Lugar':'Quito',     'Num. dormitorios':1,'Num. banos':1.0,'Area':50.0, 'Num. garages':0.0},
])
preds = best_pipe.predict(ejemplos)
ejemplos['Precio_Predicho'] = preds.round(2)
ejemplos